## GPU ENGINE. UVM and streaming for Single GPU
Like polars streaming engine, for GPU an streaming memory engine is in beta, aslo taking advantage of UVM (unified virutal memory) combining machine RAM and GPU VRAM.

[UVM, data larger than GPU memory post (NVIDIA blog)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars blog)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)

[Polars GPU Streaming post (NVIDIA blog)](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)

In [1]:
from IPython import get_ipython
import polars as pl
import os

zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [ ]:
df.head().collect(engine="streaming")

## GPU in memory

In [12]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough", "total_amount"]).head().collect(
    engine=gpu_in_mem_engine
)  # RuntimeError: parallel_for failed: cudaErrorMemoryAllocation: out of memory

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
# This is the manual configuration released in the first UVM post on polars and NVIDIA post.
import rmm

mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
gpu_streaming_engine = pl.GPUEngine(memory_resource=mr, raise_on_fail=True)


# The new simplified version is passing streaming executor to the GPU engine. This is the default setting if no executor is specified
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "1"  # This is the default config
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_streaming_engine)

## Multi GPU Streaming. Currently in experimental stage.
#### NOTE: Only run this part in an environment where are multiple GPUs. Kaggle can be a free option to test it.

In [1]:
# !uv pip install dask_cuda # Dask Cuda client needed

In [ ]:
## Multi GPU Streaming
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

client = Client(LocalCUDACluster())

multi_gpu_streaming = pl.GPUEngine(
    executor="streaming",
    executor_options={"scheduler": "distributed"},  # For a single machine, the executor option is synchronous
)

In [ ]:
%%time
df.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=multi_gpu_streaming)